# UrduStack — Train Risk Scorer (LoRA XLM-RoBERTa on PURUTT)

Run this notebook in Google Colab free-tier GPU (T4).

**What it does**
- Fine-tunes `xlm-roberta-base` on a 10–15k sample of PURUTT using LoRA.
- Saves a small LoRA adapter (`models/risk_lora/`), not the full model.
- Computes a temperature-scaling value from the validation set and writes it to `models/temperature.txt`.

**Expected total runtime on free Colab T4**
- Setup + installs: ~3–5 min
- Model download: ~2–5 min
- Training (3 epochs, 15k samples, batch 16): ~15–30 min
- Temperature calibration + test eval: ~1–2 min
- **Total: ~25–45 minutes**

**Before running**
- Upload `PURUTT.csv` to Colab when asked, **or**
- Mount Drive (cell 3) and place it at `/content/drive/MyDrive/UrduStack/data/raw/PURUTT.csv`.

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Install dependencies
# We keep the list minimal. Colab already has torch installed.
!pip install -q transformers datasets peft accelerate scikit-learn pandas

In [ ]:
# Optional: mount Google Drive so the trained adapter is saved permanently
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the UrduStack repo (or upload the training script manually)
!git clone https://github.com/munazat/UrduStack.git
%cd UrduStack

In [ ]:
# Upload / copy PURUTT.csv if it is not already in the repo
from google.colab import files
import shutil, os

os.makedirs('data/raw', exist_ok=True)

# If you mounted Drive and placed the CSV there, copy it over first
drive_csv = '/content/drive/MyDrive/UrduStack/data/raw/PURUTT.csv'
if os.path.exists(drive_csv) and not os.path.exists('data/raw/PURUTT.csv'):
    shutil.copy(drive_csv, 'data/raw/PURUTT.csv')
    print('Copied PURUTT.csv from Drive')

if not os.path.exists('data/raw/PURUTT.csv'):
    print('Please upload PURUTT.csv')
    uploaded = files.upload()
    for name in uploaded:
        if name.endswith('.csv'):
            shutil.move(name, 'data/raw/PURUTT.csv')
            break
else:
    print('PURUTT.csv already present')

In [ ]:
# Inspect the dataset
import pandas as pd
df = pd.read_csv('data/raw/PURUTT.csv')
print(df.head())
print(df.columns.tolist())
print(df['label'].value_counts())

In [ ]:
# Train
!python scripts/train_risk_model.py \
  --data_path data/raw/PURUTT.csv \
  --output_dir models/risk_lora \
  --max_samples 15000 \
  --val_samples 2000 \
  --test_samples 2000 \
  --num_epochs 3 \
  --batch_size 16

In [ ]:
# Check outputs
import os
print('Adapter files:', os.listdir('models/risk_lora'))
if os.path.exists('models/temperature.txt'):
    print('Temperature:', open('models/temperature.txt').read().strip())

In [ ]:
# Optional: push adapter to Hugging Face Hub
# !huggingface-cli login
# !python scripts/train_risk_model.py \
#   --data_path data/raw/PURUTT.csv \
#   --output_dir models/risk_lora \
#   --push_to_hub \
#   --hub_model_id your-username/urdustack-risk-lora

In [ ]:
# Optional: copy results to Drive (only if Drive was mounted)
import os, shutil

drive_dest = '/content/drive/MyDrive/urdustack_models'
if os.path.exists('/content/drive/MyDrive'):
    shutil.copytree('models', drive_dest, dirs_exist_ok=True)
    print('Copied models to', drive_dest)
else:
    print('Drive not mounted — skipping copy. Use the Files panel to download models/ manually.')